In [1]:
import pandas as pd
import numpy as np
import os

In [ ]:
#Import the top BPM and WPM tables
#These are for BPMs and WPMs that met a p-value threshold of 0.05 for at least 5 out of 10 runs and a FDR threshold of 0.25 for at least 1 out of 10 runs
top_bpms = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/BPM_pval_filter.xlsx', usecols= 'B:D')
top_wpms = pd.read_excel('C:/Users/sophi/Documents/Plan B Project/k10 new results/WPM_pval_filter.xlsx', usecols = 'B:C')

In [3]:
#Generate nums based on the number of case/control groups used for BridGE runs.
nums = np.arange(1,11)

In [4]:
#Open the output results files and add which run its from
for i in nums:
    #Make sure sheet exists
    target_sheets = ['output_bpm_table', 'output_wpm_table']

    xlsx = pd.ExcelFile(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_ssM_mhygessi_combined_R0.xls')
    sheets = {name: pd.read_excel(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_ssM_mhygessi_combined_R0.xls', sheet_name=name)
            for name in xlsx.sheet_names}
    #if sheet exists, add a column stating which run it's from on the sheet
    for sheet in target_sheets:
        if sheet in sheets:
            bridge_out = sheets[sheet]
            #add run column
            bridge_out['Run_Num'] = i
            sheets[sheet] = bridge_out
        else:
            print('sheets not found')
            #Write to output
    with pd.ExcelWriter(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx', engine="openpyxl") as writer:
        for name, df in sheets.items():
            df.to_excel(writer, sheet_name=name, index=False)

sheets not found
sheets not found
sheets not found
sheets not found
sheets not found


In [17]:
#Combine the BPMs to make one giant excel spreadsheet
dfs = []
bpm_target = 'output_bpm_table'
for i in nums:
    xls = pd.ExcelFile(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx')
    #make sure sheet exists
    if bpm_target in xls.sheet_names:
        df = pd.read_excel(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx', sheet_name=bpm_target)
        dfs.append(df)

#Put all the dfs together
combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
#rename combined_df columns, so you can merge them later
combined_df = combined_df.rename(columns={'path1names':'path1', 'path2names':'path2', 'eff_bpm':'disease association'})
key_cols = ["disease association", "path1", "path2"]

# rows in combined that match those top BPMs
matched = combined_df.merge(top_bpms, on=key_cols, how="inner")

#write to output
matched.to_csv('Output_Results_Table_BPM.csv')

In [8]:
#Combine the WPMs to make one giant excel spreadsheet
dfs = []
wpm_target = 'output_wpm_table'
for i in nums:
    xls = pd.ExcelFile(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx')
    #make sure sheet exists
    if wpm_target in xls.sheet_names:
        df = pd.read_excel(f'C:/Users/sophi/Documents/Plan B Project/CRC_g{i} results/output_results_with_runs.xlsx', sheet_name=wpm_target)
        dfs.append(df)

#Put all the dfs together
combined_df = pd.concat(dfs, ignore_index=True)

#rename combined_df columns, so you can merge them later
combined_df = combined_df.rename(columns={'eff_wpm':'disease association'})

key_cols = ["disease association", "pathway"]

# rows in combined that match those top WPMs
matched = combined_df.merge(top_wpms, on=key_cols, how="inner")

matched.to_csv("Output_Results_Table_WPM.csv")